# Paper 2: Compulsory citizenship behavior, LMX, conscientiousness and IWP

This notebook reproduces H1–H7, runs the **CCB² sensitivity checks** for H2, H4 and H7, and redraws **Figures 1–7** and **Supplementary Figure S4**.

**Before you run it**
1. Install the packages once (in Anaconda Prompt): `pip install pandas numpy scipy statsmodels matplotlib`
2. Keep `CCB_Two_Phase_Data.csv` in your **Downloads** folder, or change the path in the *Data location* cell.
3. Run all cells (*Kernel → Restart & Run All*). The bootstrap takes about 15–60 seconds.

The *Variables* cell checks the file against the manuscript (N = 391, CCB mean = 3.051 and so on) and flags any mismatch. Outputs are saved to a folder called **Paper2_outputs** next to the data file.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

## Settings and data location
Change `DATA` if your file is somewhere other than Downloads.

In [ ]:
N_BOOT = 10_000
SEED = 2026


DATA = Path.home() / "Downloads" / "CCB_Two_Phase_Data.csv"
# e.g. DATA = Path("data") / "CCB_Two_Phase_Data.csv"
if not DATA.exists():
    raise FileNotFoundError(f"Data file not found: {DATA} - edit the DATA line above.")
OUT = DATA.parent / "Paper2_outputs"
OUT.mkdir(exist_ok=True)

d = pd.read_csv(DATA)
report = []
def say(*a):
    line = " ".join(str(x) for x in a)
    print(line)
    report.append(line)

## Variables
Builds the scale scores from the items if the file has none, checks the file against the manuscript, then standardizes the continuous predictors (sample mean, N-denominator SD). Graduate education is the reference category.

In [ ]:
# Tidy column names and build the scale scores from the items if they are missing
d.columns = d.columns.str.strip().str.replace("\ufeff", "", regex=False)
ITEMS = {"CCB": 5, "LMX1": 7, "LMX2": 7, "TP": 5, "CP": 8, "CWB": 5,
         "SE": 10, "OR": 10, "DU": 10, "AS": 10, "SD": 10, "CA": 10}
missing_items = [f"{k}_{i}" for k, n in ITEMS.items() for i in range(1, n + 1) if f"{k}_{i}" not in d.columns]
built = []
for k, n in ITEMS.items():
    if f"{k}_mean" not in d.columns:
        cols = [f"{k}_{i}" for i in range(1, n + 1)]
        if any(c not in d.columns for c in cols):
            raise KeyError(f"Cannot build {k}_mean - item columns missing: "
                           f"{[c for c in cols if c not in d.columns]}.\nColumns in your file: {list(d.columns)}")
        d[f"{k}_mean"] = d[cols].mean(axis=1)
        built.append(f"{k}_mean")
if "CONSC_mean" not in d.columns:
    d["CONSC_mean"] = d[[f"{k}_mean" for k in ["SE", "OR", "DU", "AS", "SD", "CA"]]].mean(axis=1)
    built.append("CONSC_mean")
if "IWP_mean" not in d.columns:
    d["IWP_mean"] = (d.TP_mean + d.CP_mean + (6 - d.CWB_mean)) / 3
    built.append("IWP_mean")
if built:
    say("Scale scores computed from items:", ", ".join(built))
d = d.dropna(subset=["CCB_mean", "LMX1_mean", "LMX2_mean", "CONSC_mean", "IWP_mean"]).reset_index(drop=True)

# Sanity check against the manuscript (N = 391, CCB M = 3.051, conscientiousness M = 3.317)
chk = {"N": (len(d), 391), "CCB mean": (round(d.CCB_mean.mean(), 3), 3.051),
       "Conscientiousness mean": (round(d.CONSC_mean.mean(), 3), 3.317), "IWP mean": (round(d.IWP_mean.mean(), 3), 3.529)}
for k, (got, want) in chk.items():
    flag = "OK" if abs(got - want) < 0.002 else "<-- DIFFERS from manuscript"
    say(f"  check {k:<24} {got}  (manuscript {want})  {flag}")
if any(abs(g - w) >= 0.002 for g, w in chk.values()):
    say("  WARNING: this file does not match the manuscript sample. If conscientiousness differs, the "
        "negatively keyed IPIP items may not be reverse-scored yet in this file.")
def z(s):                       # standardize with sample mean and N-denominator SD
    return (s - s.mean()) / s.std(ddof=0)

v = pd.DataFrame({
    "ccb":   z(d.CCB_mean),
    "lmx1":  z(d.LMX1_mean),
    "lmx2z": z(d.LMX2_mean),
    "consc": z(d.CONSC_mean),
    "exp":   z(d.Total_Experience_Years),
    "torg":  z(d.Tenure_Org_Months),
    "tsup":  z(d.Tenure_Supervisor_Months),
    "fem":   d.Gender_Female,
    "mar":   d.Marital_Status_Married,
    "pg":    (d.Education_Level == 2).astype(int),   # graduate (1) = reference
    "doc":   (d.Education_Level == 3).astype(int),
    "lmx2":  d.LMX2_mean,
    "iwp":   d.IWP_mean,
    "tp":    d.TP_mean,
    "cp":    d.CP_mean,
    "cwb":   d.CWB_mean,
})
v["ccb2"] = v.ccb ** 2
v["ccb_x_c"] = v.ccb * v.consc
v["ccb2_x_c"] = v.ccb2 * v.consc
CTRL = ["fem", "pg", "doc", "mar", "exp", "torg", "tsup"]
SD_LMX2 = d.LMX2_mean.std(ddof=0)          # fixed mediator SD (.927)
N = len(v)

def ols(y, xs, data=v):
    X = sm.add_constant(data[xs])
    return sm.OLS(data[y], X).fit(cov_type="HC3")

def line(name, m, term):
    ci = m.conf_int().loc[term]
    say(f"  {name:<34} b = {m.params[term]: .3f}  SE = {m.bse[term]:.3f}  "
        f"p = {m.pvalues[term]:.3f}  95% CI [{ci[0]: .3f}, {ci[1]: .3f}]  R2 = {m.rsquared:.3f}")

say(f"Data: {DATA}  (N = {N})")
say("=" * 100)

## H1 and H2

In [ ]:
say("H1  CCB (T1) -> LMX (T2), controlling LMX (T1)")
m_h1 = ols("lmx2", ["ccb", "lmx1"] + CTRL); line("CCB", m_h1, "ccb")
say("H2  LMX (T2) -> IWP")
m_h2 = ols("iwp", ["lmx2z", "lmx1", "ccb"] + CTRL); line("LMX T2", m_h2, "lmx2z")

## H3: strict curvilinear tests (Lind & Mehlum, 2010)

In [ ]:
say("H3  Strict curvilinear tests (Lind & Mehlum, 2010)")
xlo, xhi = v.ccb.min(), v.ccb.max()
h3 = {}
for y, lab in [("tp", "Task performance"), ("cp", "Contextual performance"),
               ("cwb", "Counterproductive work behavior"), ("iwp", "IWP composite")]:
    m = ols(y, ["ccb", "ccb2"] + CTRL)
    lin = ols(y, ["ccb"] + CTRL)
    b1, b2 = m.params.ccb, m.params.ccb2
    V = m.cov_params()
    def slope(x):
        s = b1 + 2 * b2 * x
        se = np.sqrt(V.loc["ccb", "ccb"] + 4 * x**2 * V.loc["ccb2", "ccb2"] + 4 * x * V.loc["ccb", "ccb2"])
        return s, s / se
    sl, tl = slope(xlo); su, tu = slope(xhi)
    # one-sided p in the direction implied by the curve
    if b2 < 0:
        pl, pu = stats.norm.sf(tl), stats.norm.cdf(tu)
    else:
        pl, pu = stats.norm.cdf(tl), stats.norm.sf(tu)
    tp_ = -b1 / (2 * b2)
    strict = (m.pvalues.ccb2 < .05) and (xlo < tp_ < xhi) and pl < .05 and pu < .05
    shape = ("Inverted-U" if b2 < 0 else "U-shape") if strict else "Not supported"
    h3[y] = m
    say(f"  {lab:<32} b(CCB2) = {b2: .3f}  SE = {m.bse.ccb2:.3f}  p = {m.pvalues.ccb2:.3f}  "
        f"R2 = {m.rsquared:.3f}  dR2 = {m.rsquared - lin.rsquared:.3f}  lower slope = {sl: .3f} (p1 = {pl:.3f})  "
        f"upper slope = {su: .3f} (p1 = {pu:.3f})  turning z = {tp_: .3f}  -> {shape}")

## H5: conscientiousness × CCB on later LMX

In [ ]:
say("H5  CCB x conscientiousness -> LMX (T2)")
m_h5 = ols("lmx2", ["ccb", "consc", "ccb_x_c", "lmx1"] + CTRL); line("CCB x conscientiousness", m_h5, "ccb_x_c")
m_h5_base = ols("lmx2", ["ccb", "consc", "lmx1"] + CTRL)
say(f"  dR2 for interaction = {m_h5.rsquared - m_h5_base.rsquared:.3f}")
V5 = m_h5.cov_params()
for w, lab in [(-1, "low (-1 SD)"), (0, "mean"), (1, "high (+1 SD)")]:
    s = m_h5.params.ccb + w * m_h5.params.ccb_x_c
    se = np.sqrt(V5.loc["ccb", "ccb"] + w**2 * V5.loc["ccb_x_c", "ccb_x_c"] + 2 * w * V5.loc["ccb", "ccb_x_c"])
    say(f"  simple slope at {lab:<13} = {s: .3f}  p = {2 * stats.norm.sf(abs(s / se)):.3f}")

## H6: conscientiousness and the CCB–IWP curve

In [ ]:
say("H6  Conscientiousness moderation of the CCB-IWP curve")
m_h6 = ols("iwp", ["ccb", "ccb2", "consc", "ccb_x_c", "ccb2_x_c"] + CTRL)
m_h6_base = ols("iwp", ["ccb", "ccb2", "consc"] + CTRL)
line("CCB x conscientiousness", m_h6, "ccb_x_c"); line("CCB2 x conscientiousness", m_h6, "ccb2_x_c")
try:
    w6 = m_h6.wald_test("ccb_x_c = 0, ccb2_x_c = 0", scalar=True, use_f=False)
except TypeError:                      # older statsmodels has no "scalar" argument
    w6 = m_h6.wald_test("ccb_x_c = 0, ccb2_x_c = 0", use_f=False)
say(f"  joint robust Wald chi2(2) = {float(np.squeeze(w6.statistic)):.3f}  p = {float(np.squeeze(w6.pvalue)):.3f}  "
    f"dR2 = {m_h6.rsquared - m_h6_base.rsquared:.3f}")

## Bootstrap engine and H4 / H7
10,000 resamples with BCa intervals. Small third-decimal differences from the manuscript come from the random seed.

In [ ]:
def mat(cols):
    return np.column_stack([np.ones(N)] + [v[c].to_numpy(float) for c in cols])

def coefs(X, y, idx):
    return np.linalg.lstsq(X[idx], y[idx], rcond=None)[0]

def bca(est, boot, jack, level=.95):
    boot = boot[np.isfinite(boot)]
    z0 = stats.norm.ppf((boot < est).mean() + 0.5 * (boot == est).mean())
    jm = jack.mean()
    acc = ((jm - jack) ** 3).sum() / (6 * (((jm - jack) ** 2).sum()) ** 1.5)
    out = []
    for q in [(1 - level) / 2, 1 - (1 - level) / 2]:
        zq = stats.norm.ppf(q)
        out.append(np.quantile(boot, stats.norm.cdf(z0 + (z0 + zq) / (1 - acc * (z0 + zq)))))
    return out

def run_indirect(a_cols, b_cols, label, conditional=False):
    """a-path: raw LMX2 on a_cols; b-path: IWP on b_cols (which contain lmx2z).
    Indirect = a * b / SD(LMX2). With conditional=True the a-model contains
    ccb, consc and ccb_x_c, and conditional effects at -1/0/+1 SD plus the index are returned."""
    Xa, Xb = mat(a_cols), mat(b_cols)
    ya, yb = v.lmx2.to_numpy(float), v.iwp.to_numpy(float)
    ia, ib = 1 + a_cols.index("ccb"), 1 + b_cols.index("lmx2z")
    ix = 1 + a_cols.index("ccb_x_c") if conditional else None

    def effects(idx):
        ca, cb = coefs(Xa, ya, idx), coefs(Xb, yb, idx)
        b = cb[ib] / SD_LMX2
        if not conditional:
            return np.array([ca[ia] * b])
        return np.array([(ca[ia] - ca[ix]) * b, ca[ia] * b, (ca[ia] + ca[ix]) * b, ca[ix] * b])

    full = np.arange(N)
    est = effects(full)
    rng = np.random.default_rng(SEED)
    boot = np.array([effects(rng.integers(0, N, size=N)) for _ in range(N_BOOT)])
    jack = np.array([effects(np.delete(full, i)) for i in range(N)])
    names = ["indirect"] if not conditional else ["low (-1 SD)", "mean", "high (+1 SD)", "index of mod. med."]
    rows = []
    say(f"{label}")
    for k, nm in enumerate(names):
        lo, hi = bca(est[k], boot[:, k], jack[:, k])
        say(f"  {nm:<20} = {est[k]: .3f}  boot SE = {boot[:, k].std(ddof=1):.3f}  95% BCa CI [{lo: .3f}, {hi: .3f}]")
        rows.append({"model": label, "effect": nm, "estimate": round(est[k], 3),
                     "boot_SE": round(boot[:, k].std(ddof=1), 3), "BCa_lower": round(lo, 3), "BCa_upper": round(hi, 3)})
    return rows

A_H4 = ["ccb", "lmx1"] + CTRL
B_H4 = ["lmx2z", "lmx1", "ccb"] + CTRL
A_H7 = ["ccb", "consc", "ccb_x_c", "lmx1"] + CTRL
B_H7 = ["ccb", "lmx1", "consc", "lmx2z"] + CTRL

say("=" * 100)
say(f"Bootstrap: {N_BOOT:,} resamples, seed {SEED}, BCa intervals (jackknife acceleration)")
primary = run_indirect(A_H4, B_H4, "H4  Indirect CCB -> LMX2 -> IWP (primary)")
primary += run_indirect(A_H7, B_H7, "H7  Conditional indirect + index (primary)", conditional=True)

## NEW: CCB² sensitivity check
Adds CCB² to the IWP (outcome) equations of H2, H4 and H7. Saved as `sensitivity_CCBsq.csv` for the supplement.

In [ ]:
say("=" * 100)
say("SENSITIVITY: CCB^2 added to the outcome (IWP) equations")
m_h2s = ols("iwp", ["lmx2z", "lmx1", "ccb", "ccb2"] + CTRL); line("H2 + CCB2: LMX T2", m_h2s, "lmx2z")
line("H2 + CCB2: CCB2", m_h2s, "ccb2")
m_h7bs = ols("iwp", ["ccb", "ccb2", "lmx1", "consc", "lmx2z"] + CTRL); line("H7 outcome + CCB2: LMX T2", m_h7bs, "lmx2z")
sens = run_indirect(A_H4, B_H4 + ["ccb2"], "H4 with CCB2 in outcome equation")
sens += run_indirect(A_H7, B_H7 + ["ccb2"], "H7 with CCB2 in outcome equation", conditional=True)

tab = pd.DataFrame(primary + sens)
tab.to_csv(OUT / "sensitivity_CCBsq.csv", index=False)
(OUT / "results_summary.txt").write_text("\n".join(report), encoding="utf-8")

## Figures
Saved as 300 dpi PNG and vector PDF.

In [ ]:
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
                     "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.linewidth": 0.8, "savefig.dpi": 300, "savefig.bbox": "tight"})
INK, MUTED, BLUE = "#0b0b0b", "#9a9893", "#2a78d6"
SEQ = {"Low conscientiousness (−1 SD)": ("#86b6ef", "--"),
       "Mean conscientiousness":        ("#2a78d6", "-"),
       "High conscientiousness (+1 SD)": ("#104281", "-.")}
XLAB = "Compulsory citizenship behavior (Time 1)"
mu, sd = d.CCB_mean.mean(), d.CCB_mean.std(ddof=0)
raw_grid = np.linspace(1, 5, 200)
zg = (raw_grid - mu) / sd
rng = np.random.default_rng(1)
jit = lambda s: s + rng.uniform(-0.08, 0.08, len(s))

def predict(m, overrides):
    """Predicted values + 95% CI with all other predictors at their sample means."""
    cols = list(m.params.index)
    X = np.tile(v[cols[1:]].mean().to_numpy(), (len(zg), 1))
    X = np.column_stack([np.ones(len(zg)), X])
    for c, val in overrides.items():
        X[:, cols.index(c)] = val
    fit = X @ m.params.to_numpy()
    se = np.sqrt(np.einsum("ij,jk,ik->i", X, m.cov_params().to_numpy(), X))
    return fit, fit - 1.96 * se, fit + 1.96 * se

def finish(ax, ylab, name):
    ax.set_xlabel(XLAB); ax.set_ylabel(ylab)
    ax.set_xlim(0.8, 5.2); ax.set_ylim(0.8, 5.2)
    ax.set_xticks([1, 2, 3, 4, 5]); ax.set_yticks([1, 2, 3, 4, 5])
    fig = ax.figure
    fig.savefig(OUT / f"{name}.png"); fig.savefig(OUT / f"{name}.pdf"); plt.close(fig)

def single_curve(m, ycol, ylab, name, quad):
    fig, ax = plt.subplots(figsize=(5.2, 3.9))
    ax.scatter(jit(d.CCB_mean), d[ycol], s=9, color=MUTED, alpha=.35, linewidths=0, label="Observed (jittered)")
    ov = {"ccb": zg, "ccb2": zg**2} if quad else {"ccb": zg}
    fit, lo, hi = predict(m, ov)
    ax.fill_between(raw_grid, lo, hi, color=BLUE, alpha=.18, linewidth=0, label="95% CI")
    ax.plot(raw_grid, fit, color=BLUE, lw=2, label="Adjusted fit")
    ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=8.5)
    finish(ax, ylab, name)

# Figure 1: H1
single_curve(m_h1, "LMX2_mean", "Leader–member exchange (Time 2)", "Figure1", quad=False)
# Figures 2-4: H3
single_curve(h3["iwp"], "IWP_mean", "Individual work performance (composite)", "Figure2", quad=True)
single_curve(h3["cp"], "CP_mean", "Contextual performance", "Figure3", quad=True)
single_curve(h3["cwb"], "CWB_mean", "Counterproductive work behavior", "Figure4", quad=True)
single_curve(h3["tp"], "TP_mean", "Task performance", "FigureS4", quad=True)

# Figure 5: H5 conditional lines
fig, ax = plt.subplots(figsize=(5.2, 3.9))
for (lab, (col, ls)), w in zip(SEQ.items(), [-1, 0, 1]):
    fit, lo, hi = predict(m_h5, {"ccb": zg, "consc": w, "ccb_x_c": zg * w})
    ax.fill_between(raw_grid, lo, hi, color=col, alpha=.12, linewidth=0)
    ax.plot(raw_grid, fit, color=col, ls=ls, lw=2, label=lab)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=8.5)
finish(ax, "Leader–member exchange (Time 2)", "Figure5")

# Figure 6: H6 conditional curves
fig, ax = plt.subplots(figsize=(5.2, 3.9))
for (lab, (col, ls)), w in zip(SEQ.items(), [-1, 0, 1]):
    fit, _, _ = predict(m_h6, {"ccb": zg, "ccb2": zg**2, "consc": w, "ccb_x_c": zg * w, "ccb2_x_c": zg**2 * w})
    ax.plot(raw_grid, fit, color=col, ls=ls, lw=2, label=lab)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=8.5)
finish(ax, "Individual work performance (composite)", "Figure6")

# Figure 7: H7 conditional indirect associations
h7 = [r for r in primary if r["model"].startswith("H7")][:3]
fig, ax = plt.subplots(figsize=(4.6, 3.6))
xs = np.arange(3)
est = np.array([r["estimate"] for r in h7])
err = np.array([[r["estimate"] - r["BCa_lower"] for r in h7], [r["BCa_upper"] - r["estimate"] for r in h7]])
ax.axhline(0, color=MUTED, lw=1, ls="--")
ax.errorbar(xs, est, yerr=err, fmt="o", color=BLUE, ms=7, capsize=5, lw=1.6)
ax.set_xticks(xs); ax.set_xticklabels(["Low (−1 SD)", "Mean", "High (+1 SD)"])
ax.set_xlim(-0.5, 2.5)
ax.set_xlabel("Conscientiousness")
ax.set_ylabel("Conditional indirect association with IWP")
fig.savefig(OUT / "Figure7.png"); fig.savefig(OUT / "Figure7.pdf"); plt.close(fig)

print("\nAll outputs saved to:", OUT)

## Preview the figures

In [ ]:
from IPython.display import Image, display
for f in ["Figure1","Figure2","Figure3","Figure4","Figure5","Figure6","Figure7","FigureS4"]:
    print(f); display(Image(filename=str(OUT / f"{f}.png"), width=520))